<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_SIR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 9b — SIR

This notebook runs only the `sir` SBIBM task. It delegates the ML experiment to the shared Exercise-9b runner, so it retains exactly the same flow topology, four-member flow ensemble, ten-member plain classifier ensemble, pure-CE objective, 250-epoch stepped learning-rate campaign, checkpoint validation, metrics, and artifact schema as the other task notebooks.

## Important: start from a fresh Colab runtime

If you previously attempted the Julia installation, select **Runtime → Restart session**, reopen this notebook if necessary, and then choose **Runtime → Run all**. No mid-notebook restart and no Julia installation are required. The source checkout is runtime-local; models, simulation banks, results, and standalone figure scripts are saved under `MyDrive/hybrid_nsbi_ml/exercise_9b_SBIBM`.

## Why this notebook does not use `diffeqtorch`

`sbibm==1.1.0` implements SIR through the now-legacy `diffeqtorch`/PyJulia/SciML stack, which is not compatible with current Colab's Julia and Python runtime. Here we replace **only the numerical ODE solver**. The statistical task is unchanged:

$$\dot S=-\beta SI/N,\qquad \dot I=\beta SI/N-\gamma I,\qquad \dot R=\gamma I,$$

with the official population and initial state, the official LogNormal prior for $(\beta,\gamma)$, infected-population summaries at days $0,17,\ldots,153$, and the official Binomial observation model with 1000 trials. Official observations and official reference-posterior samples are still read directly from `sbibm`.

The replacement is a vectorized fourth-order Runge–Kutta integration with $\Delta t=0.1$ day. Before any simulations or training, the notebook compares it with a high-accuracy DOP853 solution at central and crossed two-standard-deviation prior points. The run stops unless the maximum absolute error in the infected fraction is below $5\times10^{-7}$, corresponding to less than $5\times10^{-4}$ in the expected Binomial count. The backend identifier and numerical audit are written into the task status JSON and results table.

After this finishes, run `Exercise_9b_SBIBM.ipynb` to include SIR in the partial or complete comparison.

In [ ]:
TASK_NAME = "sir"
PROFILE = "PAPER"       # SMOKE | TUTORIAL | PAPER
BASE_SEED = 29082026
LOAD_IF_AVAILABLE = False
FAIL_ON_TASK_ERROR = True

import os
os.environ["EX9B_TASK"] = TASK_NAME
os.environ["EX9B_PROFILE"] = PROFILE
os.environ["EX9B_SEED"] = str(BASE_SEED)
os.environ["EX9B_LOAD_IF_AVAILABLE"] = "1" if LOAD_IF_AVAILABLE else "0"
os.environ["EX9B_FAIL_ON_TASK_ERROR"] = "1" if FAIL_ON_TASK_ERROR else "0"
os.environ["EX9B_SIR_BACKEND"] = "python_rk4"

## Run the audited SIR task

The shared engine installs the lightweight Python dependencies, activates and audits the SIR compatibility backend, generates or loads the exact 10,000-pair training bank, and then starts the matched JANA-versus-hybrid experiment. A line beginning with `SIR backend preflight:` must appear before training.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"

if "google.colab" in sys.modules:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    repository = Path("/content/nsbi-lhc-toolkit")
    if not (repository / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none",
            "--sparse", "--branch", BRANCH, REPO_URL, str(repository),
        ], check=True, env=clone_env)
    else:
        subprocess.run([
            "git", "-C", str(repository), "fetch", "origin", BRANCH,
        ], check=True)
        subprocess.run([
            "git", "-C", str(repository), "checkout", BRANCH,
        ], check=True)
        subprocess.run([
            "git", "-C", str(repository), "pull", "--ff-only",
            "origin", BRANCH,
        ], check=True)
    subprocess.run([
        "git", "-C", str(repository), "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    ], check=True)
    runner = (
        repository / "workshops" / "ml4hep_tifr_colab"
        / "Exercise_9b_SBIBM_Core.ipynb"
    )
else:
    candidates = [
        Path.cwd() / "Exercise_9b_SBIBM_Core.ipynb",
        Path.cwd() / "workshops" / "ml4hep_tifr_colab"
        / "Exercise_9b_SBIBM_Core.ipynb",
    ]
    runner = next((path for path in candidates if path.exists()), None)
    if runner is None:
        raise FileNotFoundError("Cannot locate Exercise_9b_SBIBM_Core.ipynb")

print("Running shared task engine:", runner)
get_ipython().run_line_magic("run", f'"{runner}"')